In [215]:
from pydantic import BaseModel, Field, RootModel
from typing import Literal, List, Any, Union, Dict, Callable, Optional
from datetime import datetime, timezone
import random, time, json, os
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain.chat_models import init_chat_model
from langchain_deepseek import ChatDeepSeek

In [216]:
class Tool:
    def __init__(
            self, name: str, 
            description: str, 
            input_schema: Dict[str, Any],
            output_schema: Dict[str, Any],
            func: Callable[..., Any]
            ):
        self.name = name
        self.description = description
        self.input_schema = input_schema
        self.output_shcema = output_schema
        self.func = func

    def __call__(self, **kwargs):
        return self.func(**kwargs)

In [217]:
class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, Tool] = {}

    def registry(self, tool: Tool):
        self.tools[tool.name] = tool

    def get(self, name: str) -> Tool:
        if name not in self.tools.keys():
            print(f"{name}工具未注册")
        return self.tools[name]
    
    def tool_lists(self):
        return [
           {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.input_schema.model_json_schema()
                }
            } 
            for tool in self.tools.values()
        ]

    def get_tool_call_args_type(self) -> Union[BaseModel]:
        input_args_models = [tool.input_schema for tool in self.tools.values()]
        tool_call_args = Union[tuple(input_args_models)]
        return tool_call_args
    
    def get_tool_names(self) -> Literal[None]:
        return Literal[*self.tools.keys()]

In [218]:
def add(a: int, b: int) -> int:
    return a + b

def mul(a: int, b: int) -> int:
    return a * b

class AddArgs(BaseModel):
    a: int
    b: int

class MulArgs(BaseModel):
    a: int
    b: int

In [219]:
registry = ToolRegistry()


add_args = {
    "name": "add",
    "description": "Add two numbers",
    "input_schema": AddArgs,
    "output_schema": {"result": "int"},
    "func": add
}

mul_args = {
    "name": "mul",
    "description": "Multiply two numbers",
    "input_schema": MulArgs,
    "output_schema": {"result": "int"},
    "func": mul
}

add_tool = Tool(**add_args)
mul_tool = Tool(**mul_args)
registry.registry(add_tool)

registry.registry(mul_tool)

In [220]:
ToolName = registry.get_tool_names()
ToolArgs = registry.get_tool_call_args_type()

class ToolCall(BaseModel):
    action: Literal["tool"]
    thought: str
    tool_name: ToolName
    tool_args: ToolArgs

class FinalAnswer(BaseModel):
    action: Literal["final"]
    answer: str

LLMResponse = RootModel[Union[ToolCall, FinalAnswer]]

In [221]:
class DeepSeek:
    def __init__(self, client, model, memory_store, tools):
        self.client = client
        self.model = model
        self.memory_store = memory_store
        self.tools = tools
        self.system_prompt = self._create_system_prompt()

    def _create_system_prompt(self) -> str:
        tools_description = json.dumps(
            self.tools.tool_lists(),
            indent=2
        )

        system_prompt = """
        你是一个对话AI agent，你可以使用外部工具。
        硬性规则（必须遵守的规则）：
        - 严禁在内部执行任何可由工具完成的操作；
        - 如果存在可执行任务任何部分的工具，必须使用该工具；
        - 严禁跳过工具，即使是简单或显而易见的步骤；
        - 严禁将多个操作合并为单个步骤，除非某个工具明确支持这样做；
        - 当且仅当没有任何工具能进一步推进任务时，你才可以生成最终答案。
        工具使用规则：
        - 每次工具调用必须执行且仅执行一个有意义的操作；
        - 如果任务需要多个操作，你必须按照顺序调用工具；
        - 如果多个工具都适用，选择最具体的哪一个。
        响应格式：
        - 你必须仅以有效的JSON格式进行响应；
        - 严禁在JSON之外包含任何解释说明；
        - 每次响应必须且仅能选择一个动作。
        工具调用格式：
        {
            "action": "tool",
            "thought": "...",
            "tool_name": "...",
            "inputs": { ... }
        }
        最终输出格式:
        {
            "action": "final",
            "answer": "..."
        }""" + "\\n\\n可使用的工具有:\\n" + tools_description
        return system_prompt
    
    def _format_openai_chat_history(self, history: list[dict]) -> list:
        formatted_history = []
        for message in history:
            if message["role"] == "user":
                formatted_history.append(
                    HumanMessage(content=message["content"])
                )
            if message["role"] == "assistant":
                formatted_history.append(
                    AIMessage(content=message["content"])
                )
            if message["role"] == "tool":
                formatted_history.append(
                    ToolMessage(content=message["content"])
                )
        return formatted_history
    
    
    def generate(self, history: list[dict]):

        history_format = self._format_openai_chat_history(history)

        messages = [
            SystemMessage(content=self.system_prompt)
        ] + history_format

        response = self.client.invoke(
            messages,
            tools=self.tools.tool_lists()
        )

        print("FULL RESPONSE:", response)
        print("CONTENT:", response.content)
        print("TOOL CALLS:", getattr(response, "tool_calls", None))

        return response

In [222]:
class Agent:
    def __init__(self, llm, tool_registry, max_steps=5):
        self.llm = llm
        self.tool_registry = tool_registry
        self.history = []
        self.max_steps = max_steps
    
    def run(self, user_input: str):
        self.history.append({"role": "user", "content": user_input})
        for step in range(self.max_steps):
            llm_output = self.llm.generate(self.history)
            print(llm_output)
            self.history.append(llm_output)
            response = llm_output

            if response.tool_calls:
                tool_call = response.tool_calls[0]
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]

                # 记录 assistant 调用工具
                self.history.append({
                    "role": "assistant",
                    "content": "",
                    "tool_calls": response.tool_calls
                })

                # 执行工具
                tool = self.tool_registry.get(tool_name)
                result = tool(**tool_args)

                # 记录 tool 返回
                self.history.append({
                    "role": "tool",
                    "tool_name": tool_name,
                    "content": str(result)
                })
                continue
            # 如果模型直接回答
            else:

                self.history.append({
                    "role": "assistant",
                    "content": response.content
                })

                return response.content
        raise RuntimeError("Agent 没有在最大步数前完成")

In [223]:
class MemoryStore:
    def __init__(self, file_path: str, max_entries: int = 50):
        self.file_path = file_path
        self.max_entries = max_entries
        self._ensure_file()

    def _ensure_file(self):
        if not os.path.exists(self.file_path):
            print(f"创建文件{self.file_path}")
            os.makedirs(os.path.dirname(self.file_path), exist_ok=True)
            with open(self.file_path, 'w') as f:
                json.dump([], f)

    def load_all(self) -> List[dict]:
        try:
            with open(self.file_path, 'r') as f:
                return json.load(f)
        except Exception:
            return []
        
    def append(self, entry: dict):
        data = self.load_all()
        data.append(entry)

        with open(self.file_path, 'w') as f:
            json.dump(data, f, indent=2)

    def get_recent(self, limit: Optional[int] = None) -> list[dict]:
        data = self.load_all()
        limit = limit or self.max_entries
        return data[-limit:]
    
    def delete_all(self):
        with open(self.file_path, 'w') as f:
            json.dump([], f)

In [224]:
memory_store = MemoryStore("./content/memory.jsonl")

In [225]:
kwargs= { 'model':"deepseek-chat",
    'temperature':0,            
    'api_key': '',
    'base_url':"https://api.deepseek.com/v1",
}

In [226]:
client = ChatDeepSeek(**kwargs)
llm = DeepSeek(client, "deepseek-chat", memory_store, registry)
agent = Agent(llm, registry)

def chat_with_model(agent: Agent):
    print("欢迎，输入'exit'退出\\n")
    while True:
        user_input = input("User:")
        if user_input.lower() in ["exit", "quit", "q"]:
            print("再见！")
            break
        try:
            response = agent.run(user_input)
            print(f"Agent: {response}")
        except RuntimeError as e:
            print(f"Agent 错误： {e}")
        # except Exception as e:
        #     print(f"Unexpected error: {e}")

In [227]:
chat_with_model(agent)

欢迎，输入'exit'退出\n
FULL RESPONSE: content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 906, 'total_tokens': 964, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 896}, 'prompt_cache_hit_tokens': 896, 'prompt_cache_miss_tokens': 10}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '12f6e03b-68f3-4188-9e58-3317364c2c9d', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019cdc6f-2648-7482-85e0-aa51c3a9d297-0' tool_calls=[{'name': 'add', 'args': {'a': 5, 'b': 2}, 'id': 'call_00_PKRUiPaofnhnPGOFXuIlPx6T', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 906, 'output_tokens': 58, 'total_tokens': 964, 'input_token_details': {'cache_read': 896}, 'output_token_details': {}}
CONTENT: 
TOOL CALLS: [{'name': 'add', 'args': {'a': 5, 'b': 2}, 'id': 'call_00_PKRUiPaof

TypeError: 'AIMessage' object is not subscriptable

In [229]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from pydantic import BaseModel
from typing import Dict, Any, Callable, List, Optional, Union
import json
import os

# ==================== 工具定义（保持不变）====================

class Tool:
    def __init__(
        self, 
        name: str, 
        description: str, 
        input_schema: type[BaseModel],
        output_schema: Dict[str, Any],
        func: Callable[..., Any]
    ):
        self.name = name
        self.description = description
        self.input_schema = input_schema
        self.output_schema = output_schema
        self.func = func

    def __call__(self, **kwargs):
        # 验证参数
        validated = self.input_schema(**kwargs)
        return self.func(**validated.model_dump())

class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, Tool] = {}

    def register(self, tool: Tool):
        self.tools[tool.name] = tool

    def get(self, name: str) -> Optional[Tool]:
        return self.tools.get(name)
    
    def tool_lists(self):
        return [
            {
                'name': tool.name,
                'description': tool.description,
                'input_schema': tool.input_schema.model_json_schema()
            }  
            for tool in self.tools.values()
        ]

def add(a: int, b: int) -> int:
    return a + b

def mul(a: int, b: int) -> int:
    return a * b

class AddArgs(BaseModel):
    a: int
    b: int

class MulArgs(BaseModel):
    a: int
    b: int

# ==================== DeepSeek 客户端（关键修改）====================

class DeepSeek:
    def __init__(self, client, model, memory_store, tools):
        self.client = client
        self.model = model
        self.memory_store = memory_store
        self.tools = tools
        self.system_prompt = self._create_system_prompt()

    def _create_system_prompt(self) -> str:
        tools_description = json.dumps(self.tools.tool_lists(), indent=2, ensure_ascii=False)
        
        system_prompt = f"""你是一个对话AI agent，你可以使用外部工具。

硬性规则（必须遵守）：
- 严禁在内部执行任何可由工具完成的操作
- 如果存在可执行任务任何部分的工具，必须使用该工具
- 严禁跳过工具，即使是简单或显而易见的步骤
- 严禁将多个操作合并为单个步骤
- 当且仅当没有任何工具能进一步推进任务时，才可以生成最终答案

工具使用规则：
- 每次工具调用必须执行且仅执行一个有意义的操作
- 如果任务需要多个操作，必须按照顺序调用工具
- 如果多个工具都适用，选择最具体的那一个

响应格式（必须严格遵守）：
- 你必须仅以有效的JSON格式进行响应
- 严禁在JSON之外包含任何解释说明、markdown标记或代码块
- 每次响应必须且仅能选择一个动作

工具调用格式：
{{"action": "tool", "thought": "你的思考过程", "tool_name": "工具名", "tool_args": {{"参数名": 值}}}}

最终输出格式:
{{"action": "final", "answer": "你的最终回答"}}

可使用的工具有:
{tools_description}

重要：直接输出JSON，不要添加 ```json 标记或其他任何内容。"""
        return system_prompt
    
    def _format_messages(self, history: list[dict]) -> list:
        """将历史记录转换为 LangChain 消息格式"""
        messages = [SystemMessage(content=self.system_prompt)]
        
        for msg in history:
            role = msg.get("role")
            content = msg.get("content")
            
            if role == "user":
                messages.append(HumanMessage(content=content))
            elif role == "assistant":
                messages.append(AIMessage(content=content))
            elif role == "tool":
                # ToolMessage 需要 tool_call_id，这里简化处理
                messages.append(SystemMessage(content=f"工具返回: {content}"))
        
        return messages
    
    def generate(self, history: list[dict]) -> str:
        """生成响应，返回 JSON 字符串"""
        messages = self._format_messages(history)
        
        # 关键修改：不使用 with_structured_output，直接调用
        response = self.client.invoke(messages)
        
        # 提取内容并清理
        content = response.content
        
        # 清理可能的 markdown 代码块
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0].strip()
        elif "```" in content:
            content = content.split("```")[1].split("```")[0].strip()
        
        # 验证 JSON 格式
        try:
            parsed = json.loads(content)
            # 确保必要字段存在
            if parsed.get("action") == "tool":
                assert "tool_name" in parsed, "Missing tool_name"
                assert "tool_args" in parsed, "Missing tool_args"
            elif parsed.get("action") == "final":
                assert "answer" in parsed, "Missing answer"
            else:
                raise ValueError(f"Unknown action: {parsed.get('action')}")
            
            return content
            
        except (json.JSONDecodeError, AssertionError) as e:
            # 如果解析失败，返回一个默认的最终响应
            print(f"JSON解析错误: {e}, 原始内容: {content}")
            return json.dumps({
                "action": "final",
                "answer": f"抱歉，我遇到了格式错误。原始响应: {content[:100]}..."
            }, ensure_ascii=False)

# ==================== Agent（修复历史记录问题）====================

class Agent:
    def __init__(self, llm, tool_registry, max_steps=5):
        self.llm = llm
        self.tool_registry = tool_registry
        self.history = []
        self.max_steps = max_steps
    
    def run(self, user_input: str) -> str:
        self.history.append({"role": "user", "content": user_input})
        
        for step in range(self.max_steps):
            # 生成 LLM 响应
            llm_output = self.llm.generate(self.history)
            
            # 解析响应
            try:
                action = json.loads(llm_output)
            except json.JSONDecodeError:
                print(f"无法解析 LLM 输出: {llm_output}")
                continue
            
            # 处理工具调用
            if action.get("action") == "tool":
                tool_name = action.get("tool_name")
                tool_args = action.get("tool_args", {})
                
                # 记录思考过程到历史
                self.history.append({
                    "role": "assistant", 
                    "content": llm_output
                })
                
                # 执行工具
                tool = self.tool_registry.get(tool_name)
                if not tool:
                    error_msg = f"工具 '{tool_name}' 未找到"
                    self.history.append({
                        "role": "tool",
                        "content": error_msg
                    })
                    continue
                
                try:
                    result = tool(**tool_args)
                    observation = json.dumps({
                        "tool_name": tool_name,
                        "result": result
                    }, ensure_ascii=False)
                except Exception as e:
                    observation = f"工具执行错误: {str(e)}"
                
                # 记录工具结果
                self.history.append({
                    "role": "tool",
                    "content": observation
                })
                continue
            
            # 处理最终答案
            elif action.get("action") == "final":
                self.history.append({
                    "role": "assistant",
                    "content": llm_output
                })
                return action.get("answer", "无答案")
            
            else:
                print(f"未知的 action 类型: {action}")
                continue
        
        raise RuntimeError("Agent 没有在最大步数前完成")

# ==================== 内存存储（修复拼写错误）====================

class MemoryStore:
    def __init__(self, file_path: str, max_entries: int = 50):
        self.file_path = file_path
        self.max_entries = max_entries
        self._ensure_file()

    def _ensure_file(self):
        if not os.path.exists(self.file_path):
            print(f"创建文件 {self.file_path}")
            os.makedirs(os.path.dirname(self.file_path), exist_ok=True)
            with open(self.file_path, 'w', encoding='utf-8') as f:
                json.dump([], f)

    def load_all(self) -> List[dict]:
        try:
            with open(self.file_path, 'r', encoding='utf-8') as f:
                return json.load(f)  # 修复：laod -> load
        except Exception:
            return []
        
    def append(self, entry: dict):
        data = self.load_all()
        data.append(entry)
        
        # 限制条目数
        if len(data) > self.max_entries:
            data = data[-self.max_entries:]

        with open(self.file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    def get_recent(self, limit: Optional[int] = None) -> List[dict]:
        data = self.load_all()
        limit = limit or self.max_entries
        return data[-limit:]
    
    def delete_all(self):
        with open(self.file_path, 'w', encoding='utf-8') as f:
            json.dump([], f)

# ==================== 初始化与运行 ====================

if __name__ == "__main__":
    # 注册工具
    registry = ToolRegistry()
    
    add_tool = Tool(
        name="add",
        description="Add two numbers",
        input_schema=AddArgs,
        output_schema={"result": "int"},
        func=add
    )
    
    mul_tool = Tool(
        name="mul",
        description="Multiply two numbers",
        input_schema=MulArgs,
        output_schema={"result": "int"},
        func=mul
    )
    
    registry.register(add_tool)
    registry.register(mul_tool)
    
    # 初始化组件
    memory_store = MemoryStore("./content/memory.json")
    
    # 注意：base_url 末尾不能有空格！
    client = ChatOpenAI(
        model="deepseek-chat",
        temperature=0,
        api_key="",  # 替换为你的 key
        base_url="https://api.deepseek.com/v1",  # 修复：去掉末尾空格
    )
    
    llm = DeepSeek(client, "deepseek-chat", memory_store, registry)
    agent = Agent(llm, registry)

    # 运行
    print("欢迎，输入 'exit' 退出\n")
    while True:
        user_input = input("User: ")
        if user_input.lower() in ["exit", "quit", "q"]:
            print("再见！")
            break
        try:
            response = agent.run(user_input)
            print(f"Agent: {response}")
        except RuntimeError as e:
            print(f"Agent 错误: {e}")
        except Exception as e:
            print(f"意外错误: {e}")

欢迎，输入 'exit' 退出

Agent: 5 + 2 = 7
再见！
